We want the KODIS dataset to be converted to this a csv to this format:

In [13]:
import pandas as pd

df = pd.DataFrame([{"kodis-id": 123, "buyer or seller": "buyer", "concatenated utterances": "Hello my name is Kaleen. I want to make an offer. What do you mean by that?"}, {"kodis-id": 123, "buyer or seller": "seller", "concatenated utterances": "Hi my name is Sarah. I don't want to give you a refund. I mean no refund"}, ])
df.to_csv("example.csv")
df

,kodis-id,buyer or seller,concatenated utterances
0,123,buyer,Hello my name is Kaleen. I want to make an off...
1,123,seller,Hi my name is Sarah. I don't want to give you ...


# Testing helper functions

In [10]:
import pandas as pd
kodis_df = pd.read_csv("DO-NOT-DISTRIBUTE-KODIS-human-human-subset.csv")
kodis_df.columns

Index(['Unnamed: 0', 'b_StartDate', 'b_EndDate', 'b_Status', 'b_Progress',
       'b_Duration (in seconds)', 'b_Finished', 'b_RecordedDate',
       'b_ResponseId', 'b_RecipientLastName',
       ...
       's_points_binary_apol', 'b_points_binary_apol', 's_points_4level_apol',
       'b_points_4level_apol', 's_points_3level_apol', 'b_points_3level_apol',
       'joint_points_binary_apol', 'joint_points_3level_apol',
       'joint_points_4level_apol', 'Integrative_Potential_COSINE'],
      dtype='object', length=419)

Remove the following utterances:
- "I Walk Away"
- "Submitted agreement: ..."
- "Accept Deal"
- "Reject Deal"

These are preset values that the participants can click, and not utterances they type in, so we want to remove them from the LIWC analysis.

In [14]:
IGNORE_PHRASES = {"I Walk Away", "Submitted agreement:", "Accept Deal", "Reject Deal"}
BUYER = "BUYER"
SELLER = "SELLER"

In [15]:
k = 3  # Turns per segment


data = []

for index, row in kodis_df.iterrows():
    kodis_id = row['Unnamed: 0']  # The column "Unnamed: 0" is the kodis-id
    conv = (
        row['s_fullChat']
        .replace("<b>", "")
        .replace("</b>", "")
        .replace("<br>", "\n")
        .replace("[Other]", BUYER)
        .replace("[You]", SELLER)
    )

    buyer_turns, seller_turns = [], []

    for turn in conv.split("\n"):
        if turn.startswith(BUYER):
            content = turn[len(BUYER) + 1 :]
            if not any(content.startswith(phrase) for phrase in IGNORE_PHRASES):
                buyer_turns.append(content)
        elif turn.startswith(SELLER):
            content = turn[len(SELLER) + 1 :]
            if not any(content.startswith(phrase) for phrase in IGNORE_PHRASES):
                seller_turns.append(content)

    # Create overlapping sliding window from conversations 
    def sliding_window(turns, role):
        for i in range(len(turns) - k + 1):
            segment = " ".join(turns[i : i + k])
            data.append({"kodis-id": kodis_id, "buyer or seller": role, "concatenated utterances": segment})

    sliding_window(buyer_turns, "buyer")
    sliding_window(seller_turns, "seller")

    
df = pd.DataFrame(data)

df.to_csv("new_processed_conversations.csv", index=False)